# AgenTekki — Tax Agent Filler (Remake)

Over the past weeks you've built tools that automatically extract tasks from meetings, generate articles, and manage inventories. Now it's time for the **fully agentic** project: an AI agent that autonomously files a Zurich tax return.

## What's different about this notebook

Previous approaches dumped **all** taxpayer documents into every LLM call. This notebook uses a smarter **Plan-then-Execute** architecture:

1. **Plan phase** — The LLM sees only document *names and descriptions* (not content). It decides which documents it actually needs and outlines a strategy.
2. **Execute phase** — The LLM receives only the *selected* documents and produces the field mappings.

This is closer to how real AI agents work: **perceive → plan → retrieve → act**.

You will use **OpenRouter** to access LLMs via a unified API.

## Learning Objectives

By the end of this notebook you will:
- Understand the **Plan-Execute** pattern in AI agents
- Build and use an **LLM service abstraction** with a provider pattern
- Implement **selective document retrieval** (the LLM chooses what to read)
- **Visualize** agent decision-making (plans, document selection, reasoning)
- Wire a **full autonomous pipeline**: scan → plan → retrieve → fill → navigate → submit

## 1. Setup

In [ ]:
#@title Clone Repository & Install Dependencies (click to run) { display-mode: "form" }

import os, sys
os.chdir("/content")

!rm -rf /content/project3-agentic-tax-filler
!git clone -b remake-adrian https://github.com/eth-bmai-fs26/project3-agentic-tax-filler.git /content/project3-agentic-tax-filler

# Purge cached modules if re-running
for mod_name in list(sys.modules):
    if "mcp_server" in mod_name:
        del sys.modules[mod_name]

sys.path.insert(0, "/content/project3-agentic-tax-filler")

!pip install -q openai

print("Done!")

In [ ]:
#@title Build & Serve the React Frontend (click to run) { display-mode: "form" }

import subprocess, os, threading, http.server, functools, socket

FRONTEND_DIR = "/content/project3-agentic-tax-filler"
BUILD_DIR = os.path.join(FRONTEND_DIR, "EnvDemoV1", "dist")

# Pull only the EnvDemoV1 folder from the 'frontend' branch (keeps all other files from remake-adrian)
os.chdir(FRONTEND_DIR)
subprocess.run(["git", "fetch", "origin", "frontend"], check=True)
subprocess.run(["git", "checkout", "origin/frontend", "--", "EnvDemoV1"], check=True)

# Install and build
os.chdir(os.path.join(FRONTEND_DIR, "EnvDemoV1"))
subprocess.run(["npm", "install"], check=True)
subprocess.run(["npm", "run", "build"], check=True)
os.chdir("/content")

# Find a free port
def _find_free_port(start=3000, end=3020):
    for port in range(start, end):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            try:
                s.bind(("", port))
                return port
            except OSError:
                continue
    raise RuntimeError("No free port found")

PORT = _find_free_port()
handler = functools.partial(http.server.SimpleHTTPRequestHandler, directory=BUILD_DIR)
httpd = http.server.HTTPServer(("", PORT), handler)
threading.Thread(target=httpd.serve_forever, daemon=True).start()
print(f"Frontend served at http://localhost:{PORT}")

In [ ]:
# Initialize the LLM Service
from mcp_server.llm_service import LLMService
from google.colab import userdata

# Reads the OPENROUTER_API_KEY secret from Colab
# (Set it via the key icon in the left sidebar)
OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")

llm = LLMService.openai_compatible(
    model="google/gemini-3-flash-preview",
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
    default_headers={
        "HTTP-Referer": "https://github.com/eth-bmai-fs26/project3-agentic-tax-filler",
        "X-Title": "AgenTekki Tax Filler",
    },
)

# Quick test
response = llm.chat(
    system="You are a helpful assistant.",
    user="Say 'hello' in German. Reply with just the word.",
    max_tokens=20,
)
print(f"LLM says: {response}")

## 2. Simulation — Explore the Tax Portal

Before building an agent, understand what we're automating. The cell below renders the tax portal UI. Click around and explore!

**Try this:**
- Navigate through the sections using the sidebar (Personal, Income, Deductions, Wealth)
- Click "Next" to move between sub-pages
- Add rows in the Children or Bank Accounts sections
- Check the Review page to see how all data comes together
- Notice the field types: text inputs, number inputs, dropdowns, checkboxes

In [ ]:
#@title Explore the Tax Portal (click around, nothing is saved!) { display-mode: "form" }

from IPython.display import display, HTML
import os

BUILD_DIR = "/content/project3-agentic-tax-filler/EnvDemoV1/dist"
with open(os.path.join(BUILD_DIR, "index.html")) as f:
    html_content = f.read()

html_content = html_content.replace('"\/assets/', f'"http://localhost:{PORT}/assets/')
html_content = html_content.replace('"./assets/', f'"http://localhost:{PORT}/assets/')
html_content = html_content.replace('"/assets/', f'"http://localhost:{PORT}/assets/')

display(HTML(html_content))

**Reflection** (discuss with your neighbor):
- How many pages does the form have? How many sub-pages?
- What kinds of fields appear? (text, number, dropdown, checkbox, date)
- If you were an AI agent, which documents would you need for the Income page? For Deductions?
- What information can you get from the field *labels* alone, without reading any documents?

## 3. Available Documents & Guides

Before building plans, let's see what documents and tax guides the agent has access to.

In [ ]:
# Initialize the MCP Server with MockBridge (no real browser needed yet)
from mcp_server.server import MCPServer
from mcp_server.bridges.mock import MockBridge
from mcp_server.doc_catalog import build_document_catalog

REPO = "/content/project3-agentic-tax-filler"
PERSONA = "anna_meier"

mock_server = MCPServer(
    persona_folder=f"{REPO}/personas/{PERSONA}",
    guides_folder=f"{REPO}/guides",
    bridge=MockBridge(),
)

# Build the document catalog
catalog = build_document_catalog(mock_server)
print(f"Loaded {len(catalog)} documents for {PERSONA}")

In [ ]:
#@title Display document catalog { display-mode: "form" }
from IPython.display import display, HTML
rows = "".join(
    f"<tr><td><code>{d['filename']}</code></td><td>{d['description']}</td></tr>"
    for d in catalog
)
display(HTML(f'''
<h3>Documents available for {PERSONA}</h3>
<table style="border-collapse:collapse; width:100%;">
<tr style="background:#4a90d9; color:white;">
    <th style="padding:8px; text-align:left;">Filename</th>
    <th style="padding:8px; text-align:left;">Description</th>
</tr>
{rows}
</table>
<p><b>Note:</b> The Plan phase sees only this table — not the actual file contents.</p>
'''))

In [ ]:
# Fetch available tax guides
import json
from pathlib import Path

guides = mock_server.list_guides()

# Load catalog descriptions
catalog_path = Path(f"{REPO}/guides/guide_catalog.json")
guide_descs = {}
if catalog_path.exists():
    with open(catalog_path) as f:
        raw_catalog = json.load(f)
    guide_descs = {k: v["description"] for k, v in raw_catalog.items()}

print(f"Found {len(guides)} guides")

In [ ]:
#@title Display tax guides { display-mode: "form" }
from IPython.display import display, HTML

if guides:
    rows = ""
    for g in guides:
        desc = guide_descs.get(g["topic"], "<i>no description</i>")
        rows += (
            f"<tr>"
            f"<td style='padding:6px 8px;'><code>{g['title']}</code></td>"
            f"<td style='padding:6px 8px;'><code>{g['topic']}</code></td>"
            f"<td style='padding:6px 8px; font-size:13px;'>{desc}</td>"
            f"</tr>"
        )
    display(HTML(f'''
    <h3>Tax Guides</h3>
    <table style="border-collapse:collapse; width:100%;">
    <tr style="background:#2d7d2d; color:white;">
        <th style="padding:8px; text-align:left;">Title</th>
        <th style="padding:8px; text-align:left;">Topic</th>
        <th style="padding:8px; text-align:left;">Description</th>
    </tr>
    {rows}
    </table>
    <p><b>Note:</b> Use <code>server.fetch_guide(topic)</code> to read the full text of any guide.</p>
    '''))
else:
    display(HTML('''
    <h3>Tax Guides</h3>
    <p style="color:#999; font-style:italic;">No guides found.</p>
    '''))

## 4. Plan Discovery — Think Before You Act

Now that you know which documents are available, let's think about what steps are needed to fill one page of the tax form.

The agent follows a cycle: **Perceive** (scan the page) → **Plan** (decide what docs to read and how to fill) → **Act** (read docs, fill fields).

Below, you'll write plans *by hand* — the same kind of plan the LLM will later generate automatically. Use the document catalog from the previous section to decide which files you need.

In [ ]:
# Here's what a plan for the Personal page looks like:
personal_plan = {
    "_reasoning": "Personal info (name, address, DOB, AHV) comes from profile.json. "
                  "Occupation and employer come from the Lohnausweis.",
    "documents_needed": ["profile.json", "lohnausweis.txt"],
    "guides_needed": [],
    "strategy": "Extract name, address, DOB, AHV from profile.json. "
                "Get occupation and employer from Lohnausweis header.",
    "fields_to_fill": ["field-personal-firstName", "field-personal-lastName",
                       "field-personal-dateOfBirth", "field-personal-ahvNumber",
                       "field-personal-address", "field-personal-zip",
                       "field-personal-city", "field-personal-occupation",
                       "field-personal-employer"],
    "ask_user_needed": False,
}

# ============================================================
# STUDENT TODO: Fill in the income plan.
# Look at the document catalog from Section 3 and the fields
# from scan_page() in Section 5. Which documents does the LLM
# need to fill the income page? What's the strategy?
#
# Hint: income fields include gross salary, AHV/IV/EO, BVG,
#       and interest income. Where does each value come from?
# ============================================================
income_plan = {
    "_reasoning": ...,  # <-- explain which docs and why
    "documents_needed": [...],  # <-- which files contain income data?
    "guides_needed": [],
    "strategy": ...,  # <-- how will you extract each value?
    "fields_to_fill": [...],  # <-- which field locators to fill?
    "ask_user_needed": False,
}

deductions_plan = {
    "_reasoning": "Meal deductions depend on Lohnausweis field 14.1 (canteen subsidy). "
                  "Commuting costs come from the ZVV receipt. Pillar 3a contributions "
                  "come from the bank statement or pillar 3a certificate. Donations "
                  "can be found in the bank statement.",
    "documents_needed": ["lohnausweis.txt", "bank_statement.csv", "zvv_receipt.txt"],
    "guides_needed": ["berufsauslagen"],
    "strategy": "Check Lohnausweis field 14.1 for Kantinenverpflegung (Ja=1600, Nein=3200). "
                "Calculate Berufsauslagen as 3% of Nettolohn (field 12), clamped to 2000-4000. "
                "Use ZVV receipt for Fahrkosten. Find Pillar 3a contributions (max 7056). "
                "Look for Spende entries in bank statement for donations (min 100).",
    "fields_to_fill": ["field-deductions-mealCosts", "field-deductions-workExpenses",
                       "field-deductions-commutingCosts", "field-deductions-pillar3a",
                       "field-deductions-donations"],
    "ask_user_needed": False,
}

print("Plans defined for: personal, income, deductions")

In [ ]:
#@title Display plans as HTML cards { display-mode: "form" }
from IPython.display import display, HTML

for page_label, plan in [("PERSONAL", personal_plan), ("INCOME", income_plan), ("DEDUCTIONS", deductions_plan)]:
    docs = plan.get("documents_needed", [])
    docs_html = ", ".join(f"<code>{d}</code>" for d in docs) if docs else "<i>none</i>"
    guides = plan.get("guides_needed", [])
    guides_html = ", ".join(f"<code>{g}</code>" for g in guides) if guides else "<i>none</i>"
    reasoning = plan.get("_reasoning", "...")
    strategy = plan.get("strategy", "...")
    fields = plan.get("fields_to_fill", [])
    fields_html = ", ".join(f"<code>{f}</code>" for f in fields) if fields else "<i>not specified</i>"
    display(HTML(f'''
    <div style="border:2px solid #4a90d9; border-radius:8px; padding:14px; margin:10px 0;
                background:linear-gradient(135deg,#f8faff 0%,#eef3ff 100%); font-family:sans-serif;">
        <h4 style="color:#4a90d9; margin:0 0 8px 0;">{page_label}</h4>
        <div style="margin:4px 0;"><b>Reasoning:</b> {reasoning}</div>
        <div style="margin:4px 0;"><b>Documents:</b> {docs_html}</div>
        <div style="margin:4px 0;"><b>Guides:</b> {guides_html}</div>
        <div style="margin:4px 0;"><b>Strategy:</b> {strategy}</div>
        <div style="margin:4px 0;"><b>Fields to fill:</b> {fields_html}</div>
    </div>
    '''))

**Key insight:** You just did exactly what the agent's **Plan phase** will do! The difference: the LLM will write these plans automatically, seeing only the document catalog (filenames and descriptions) — not the actual file contents. This is how the agent decides what information to retrieve before making expensive document reads.

## 5. Available Tools

Let's see the tools the agent can use to interact with the tax portal.

In [ ]:
#@title Display available tools { display-mode: "form" }
tools_info = [
    ("scan_page()", "Returns all visible fields, buttons, and text on the current form page"),
    ("fill_field(locator, value)", "Enters a value into a form field identified by its locator"),
    ("click_element(locator)", "Clicks a button or navigation link"),
    ("submit_form()", "Submits the completed tax return"),
    ("list_documents()", "Lists all files in the persona's document folder"),
    ("read_document(filepath)", "Reads and extracts content from a document (.txt, .csv, .pdf, .json)"),
    ("list_guides()", "Returns available tax guide topics"),
    ("fetch_guide(url)", "Returns the full text of a specific tax guide"),
    ("ask_user(question)", "Asks the simulated taxpayer a question (penalized if unnecessary)"),
]

rows = "".join(
    f"<tr><td><code>server.{name}</code></td><td>{desc}</td></tr>"
    for name, desc in tools_info
)
display(HTML(f'''
<h3>Available Tools</h3>
<table style="border-collapse:collapse; width:100%;">
<tr style="background:#2d7d2d; color:white;">
    <th style="padding:8px; text-align:left;">Tool</th>
    <th style="padding:8px; text-align:left;">Description</th>
</tr>
{rows}
</table>
'''))

In [ ]:
# What does scan_page() actually return? Let's look at the income page.
from mcp_server.bridges.mock import MockBridge

mock = MockBridge()
mock.frontend.current_page = "income"
page_data = mock.scan_page()

import json
fillable = [el for el in page_data["elements"] if el.get("type") not in ("button", "text")]
print(f"Page: {page_data['page_name']} — {len(fillable)} fillable fields")

In [ ]:
#@title Display scan_page() results { display-mode: "form" }
from IPython.display import display, HTML

rows = "".join(
    f'<tr><td><code>{el.get("type","")}</code></td>'
    f'<td><code>{el.get("locator","")}</code></td>'
    f'<td>{el.get("label","")}</td></tr>'
    for el in fillable
)

display(HTML(f'''
<h3>Page: {page_data["page_name"]} &mdash; {len(fillable)} fillable fields</h3>
<table style="border-collapse:collapse; width:100%;">
<tr style="background:#4a90d9; color:white;">
    <th style="padding:8px; text-align:left;">Type</th>
    <th style="padding:8px; text-align:left;">Locator</th>
    <th style="padding:8px; text-align:left;">Label</th>
</tr>
{rows}
</table>
<p style="color:#666; margin-top:8px;">This is what the LLM sees when deciding which fields to fill.</p>
'''))

## 6. Think Step — Single Page Reasoning

Now we implement the **two-phase reasoning** for a single page.

### How LLM calls work: System vs. User prompts

Every call to an LLM has (at least) two parts:

| Role | What it does | Analogy |
|------|-------------|---------|
| **System prompt** | Sets the LLM's persona, rules, and output format. The model treats this as persistent instructions. | A job description pinned to the wall |
| **User prompt** | The actual question or task for *this specific call*. Changes every time. | A new task handed to the employee |

```python
response = llm.chat(
    system="You are an expert Swiss tax accountant. Always reply in JSON.",  # <-- system prompt
    user="Here are the income page fields: ... Fill them using this document: ...",  # <-- user prompt
)
```

**Why does this matter here?** Our agent makes **two LLM calls per page**, each with a *different* system prompt:

1. **Plan call** — system prompt says: *"You are a planning agent. Pick which documents to read. Do NOT fill fields yet."*
2. **Execute call** — system prompt says: *"You are a tax accountant. Given these documents, output `{locator: value}` JSON."*

The system prompt controls *what kind of task* the LLM performs. Same model, different behavior.


In [ ]:
#@title Think Step Architecture Diagram { display-mode: "form" }
from IPython.display import display, HTML

display(HTML('''
<div style="font-family: sans-serif; max-width: 620px; margin: 16px auto;">

  <!-- scan_page -->
  <div style="text-align:center; margin-bottom:4px;">
    <div style="display:inline-block; background:#e3f2fd; border:2px solid #1976d2;
                border-radius:24px; padding:8px 24px; font-weight:bold; color:#1976d2;">
      scan_page()
    </div>
  </div>
  <div style="text-align:center; color:#1976d2; font-size:22px; line-height:1;">&#9660;</div>
  <div style="text-align:center; color:#666; font-size:13px; margin-bottom:8px;">page fields</div>

  <!-- PLAN PHASE -->
  <div style="background:linear-gradient(135deg,#e8f5e9,#c8e6c9); border:2px solid #388e3c;
              border-radius:10px; padding:14px 18px; margin:0 auto 4px auto; max-width:480px;
              position:relative;">
    <div style="font-weight:bold; color:#2e7d32; font-size:15px; margin-bottom:6px;">
      1 &mdash; PLAN PHASE
    </div>
    <div style="background:#f1f8e9; border:1px dashed #66bb6a; border-radius:6px;
                padding:6px 10px; margin-bottom:8px; font-size:12px; color:#555;">
      <b>System prompt:</b> <i>&ldquo;You are a planning agent. Pick which documents to read. Do NOT fill fields yet.&rdquo;</i>
    </div>
    <div style="display:flex; justify-content:space-between; align-items:center;">
      <div style="color:#333; font-size:13px;">
        fields + doc catalog<br/>
        <span style="color:#888; font-size:12px;">(no doc content!)</span>
      </div>
      <div style="color:#2e7d32; font-size:20px; padding:0 12px;">&#10132;</div>
      <div style="background:white; border:1px solid #a5d6a7; border-radius:6px;
                  padding:6px 12px; font-size:13px; color:#333;">
        plan JSON<br/>
        <span style="color:#888; font-size:11px;">docs to read, strategy, reasoning</span>
      </div>
    </div>
  </div>
  <div style="text-align:center; color:#388e3c; font-size:22px; line-height:1;">&#9660;</div>

  <!-- SELECTIVE RETRIEVAL -->
  <div style="background:linear-gradient(135deg,#fff3e0,#ffe0b2); border:2px solid #f57c00;
              border-radius:10px; padding:14px 18px; margin:4px auto; max-width:480px;">
    <div style="font-weight:bold; color:#e65100; font-size:15px; margin-bottom:6px;">
      2 &mdash; SELECTIVE RETRIEVAL
    </div>
    <div style="display:flex; justify-content:space-between; align-items:center;">
      <div style="color:#333; font-size:13px;">
        read only the docs<br/>the plan requested
      </div>
      <div style="color:#e65100; font-size:20px; padding:0 12px;">&#10132;</div>
      <div style="background:white; border:1px solid #ffcc80; border-radius:6px;
                  padding:6px 12px; font-size:13px; color:#333;">
        doc contents
      </div>
    </div>
    <div style="margin-top:8px; font-size:12px; color:#888; font-style:italic;">
      No LLM call &mdash; just filesystem reads based on the plan
    </div>
  </div>
  <div style="text-align:center; color:#f57c00; font-size:22px; line-height:1;">&#9660;</div>

  <!-- EXECUTE PHASE -->
  <div style="background:linear-gradient(135deg,#ede7f6,#d1c4e9); border:2px solid #5e35b1;
              border-radius:10px; padding:14px 18px; margin:4px auto; max-width:480px;">
    <div style="font-weight:bold; color:#4527a0; font-size:15px; margin-bottom:6px;">
      3 &mdash; EXECUTE PHASE
    </div>
    <div style="background:#ede7f6; border:1px dashed #9575cd; border-radius:6px;
                padding:6px 10px; margin-bottom:8px; font-size:12px; color:#555;">
      <b>System prompt:</b> <i>&ldquo;You are an expert Swiss tax accountant. Given these documents, output {locator: value} JSON.&rdquo;</i>
    </div>
    <div style="display:flex; justify-content:space-between; align-items:center;">
      <div style="color:#333; font-size:13px;">
        plan + doc contents<br/>+ fields
      </div>
      <div style="color:#4527a0; font-size:20px; padding:0 12px;">&#10132;</div>
      <div style="background:white; border:1px solid #b39ddb; border-radius:6px;
                  padding:6px 12px; font-size:13px; color:#333;">
        <code>{locator: value}</code> JSON
      </div>
    </div>
  </div>
  <div style="text-align:center; color:#5e35b1; font-size:22px; line-height:1;">&#9660;</div>

  <!-- fill_field -->
  <div style="text-align:center; margin-top:4px;">
    <div style="display:inline-block; background:#fce4ec; border:2px solid #c62828;
                border-radius:24px; padding:8px 24px; font-weight:bold; color:#c62828;">
      fill_field() for each
    </div>
  </div>

  <!-- Legend -->
  <div style="margin-top:16px; padding:10px 14px; background:#f5f5f5; border-radius:8px;
              border:1px solid #ddd; font-size:12px; color:#555;">
    <b>Key idea:</b> The <b>system prompt</b> (dashed boxes) controls <i>what kind of task</i> the LLM performs.
    Same model, same API call structure &mdash; but phase 1 produces a plan while phase 3 produces field values,
    because each gets a different system prompt.
  </div>

</div>
'''))


In [ ]:
# ── GUIDE CATALOG & PLAN PHASE PROMPT ──────────────────────────

# Load guide catalog from external file
def load_guide_catalog(guides_folder):
    """Load guide descriptions from guide_catalog.json."""
    catalog_path = Path(guides_folder) / "guide_catalog.json"
    if catalog_path.exists():
        with open(catalog_path) as f:
            raw = json.load(f)
        # Flatten to {name: "description. Use when: ..."} for prompt building
        return {
            name: f"{entry['description']} Use when: {entry['use_when']}"
            for name, entry in raw.items()
        }
    return {}

GUIDE_CATALOG = load_guide_catalog(f"{REPO}/guides")
print(f"Loaded {len(GUIDE_CATALOG)} guide descriptions")


PLAN_SYSTEM_PROMPT = """You are an expert Swiss tax accountant AI.
You are planning how to fill one page of a Zurich tax return.

You will see:
1. The fields on the current form page (locators, labels, types)
2. A catalog of available documents (names and descriptions only -- NOT their content)
3. A catalog of available tax guides (names and descriptions -- NOT their content)
4. What you have already filled on previous pages

Your job: decide which documents AND which guides you need to read, and outline your strategy for filling the fields.

## Response Format (JSON only, no other text)
{
  "_reasoning": "Your step-by-step reasoning about what this page needs",
  "documents_needed": ["filename1.txt", "filename2.csv"],
  "guides_needed": ["guide-name"],
  "strategy": "Brief description of how you will fill each field",
  "fields_to_fill": ["field-locator-1", "field-locator-2"],
  "ask_user_needed": false
}

## Rules
- Only request documents that are relevant to the current page's fields.
- Request guides whenever a field involves a CALCULATION, THRESHOLD, or SPECIAL RULE (e.g. deductions, medical expenses, commuting costs).
- If no fields need filling (e.g. attachments, review), return an empty documents_needed list.
- Be specific in your strategy: mention which document contains which value.
- Do NOT guess document content -- you can only see names and descriptions.
"""



def build_plan_user_message(page_name, fields, catalog, guides_list, filled_history):
    """Build the user message for the PLAN phase."""
    # Filter to fillable fields only
    fillable = [f for f in fields if f.get("type") not in ("button", "text")]
    fields_desc = json.dumps(fillable, indent=2)

    catalog_text = "\n".join(
        f"- {d['filename']}: {d['description']}" for d in catalog
    )

    # Build guide catalog with descriptions so the LLM can make informed selections
    guides_text_lines = []
    for g in guides_list:
        desc = GUIDE_CATALOG.get(g, "")
        if desc:
            guides_text_lines.append(f"- {g}: {desc}")
        else:
            guides_text_lines.append(f"- {g}")
    guides_text = "\n".join(guides_text_lines) if guides_text_lines else "(no guides available)"

    history_lines = []
    for prev_page, prev_fields in filled_history.items():
        entries = ", ".join(f"{k}={v}" for k, v in prev_fields.items() if not k.startswith("_"))
        if entries:
            history_lines.append(f"  {prev_page}: {entries}")
    history_text = "\n".join(history_lines) if history_lines else "  (none yet)"

    # ============================================================
    # STUDENT TODO: Assemble the user message.
    #
    # You have these variables ready to use:
    #   - page_name:   name of the current form page
    #   - fields_desc: JSON string of all fillable fields
    #   - catalog_text: formatted list of documents (name: description)
    #   - guides_text:  formatted list of guides (name: description)
    #   - history_text: what was filled on previous pages
    #
    # Build an f-string that tells the LLM:
    #   1. Which page it's on
    #   2. What fields are available
    #   3. What documents exist (names only!)
    #   4. What guides exist
    #   5. What was already filled
    #
    # End with: "Respond with JSON only."
    # ============================================================
    user_msg = ...  # <-- FILL THIS IN

    return user_msg


In [ ]:
# ── EXECUTE PHASE PROMPT ───────────────────────────────────────
import json, re, time
from pathlib import Path

EXECUTE_SYSTEM_PROMPT = """You are an expert Swiss tax accountant AI filling a Zurich tax return.

You will see:
1. Your own plan (reasoning and strategy from the planning phase)
2. The actual content of the documents you requested
3. The fields on the current form page
4. Tax guides with rules and thresholds
5. What you already filled on previous pages

Your task: fill the fields on this page using the document content and your plan.

## Response Format (JSON only, no other text)
Return a JSON object mapping field locators to values:
{"_thought": "reasoning here", "field-locator-1": "value1", "field-locator-2": "value2"}

## Rules
- ONLY use locators shown in the current page fields -- NEVER guess locators.
- If a field already has the correct value, do NOT include it.
- If a field is required but you have no data, skip it (don't make up values).
- If no fields apply, return {}.
- Do NOT fill fields with empty strings, "None", "0", or placeholder text.
- Include a "_thought" key explaining your reasoning.
"""

# Section-specific prompts (appended to EXECUTE_SYSTEM_PROMPT per page)
SECTION_PROMPTS = {
    "personal": """## Personal Details
- Name, Address, DOB, AHV: Extract from profile.json.
- Parse address into street, streetNumber, zip, city.
- Marital status: "ledig", "verheiratet", "geschieden", etc.
- Occupation & Employer: From the Lohnausweis header.
- Children: Fill BOTH name AND dateOfBirth for each child in profile.json. For the `relationship` field, always use the exact value "child"
  (NOT "Kind", "Tochter", "Sohn", "daughter", or "son" — the form only
  accepts "child"). If profile.json has N children but the form shows fewer rows, include "_rows_needed": N in your response.
""",

    "income": """## Income Section
- Bruttolohn = Lohnausweis field 1 (gross salary).
- AHV/IV/EO = Lohnausweis field 9.
- BVG = Lohnausweis field 10.
- Dienstaltersgeschenke (field 6) are ALREADY INCLUDED in Bruttolohn -- do NOT add separately.
- Interest income: sum ALL "Zinsabschluss" entries from bank statement.
- Self-employment: if ANY freelance/Nebenerwerb documents exist (invoices,1099s, freelance folder), set `selfemployment.enabled` to True AND fill
  the income fields. If no freelance documents exist, leave it False.
- For married couples with two Lohnausweise: check locator names first.
  If the form has *_partner fields, fill each spouse separately.
  Otherwise, SUM the two Lohnausweise values into the single fields.
- AHV/IV/EO/ALV contributions: Lohnausweis field 9, MAIN line only
  (the "Beiträge AHV/IV/EO/ALV" line). Do NOT include the NBU sub-line.
- For married couples: SUM the main-line AHV values of both spouses.
""",

    "deductions": """## Deductions Section
- Verpflegung (meals): Lohnausweis field 14.1 Ja = CHF 1,600; Nein = CHF 3,200.
  Part-time: prorate by Pensum (e.g. 60% + "Nein" = 1920).
- Berufsauslagen Pauschale: 3% of Nettolohn (field 12), min CHF 2,000, max CHF 4,000.
  Fill BOTH `flatrate.amount` AND `weitereberufsauslagen.amount` with this value.
  For `weitereberufsauslagen.description`, use exactly: "Berufsauslagen-Pauschale (3% des Nettolohns)"
- Fahrkosten (commuting): use ZVV receipt amount.
- Pillar 3a: max CHF 7,056 (employed with BVG).
- Donations: min CHF 100, look for "Spende" in bank statement.
- Schuldzinsen: sum ALL mortgage interest from bank statement.
- Weiterbildung: only if course is related to current profession.
- Zweiverdienerabzug: if both spouses employed, fill exactly CHF 5,900 per policy_kb.docx section 2.10. Do NOT apply a percentage formula.
- Kinderbetreuungskosten: cap per child per year is CHF 10,100 per policy_kb.docx section 2.8. If receipts exceed the cap, use the cap.
- Berufsauslagen Pauschale vs. effektiv: these are ALTERNATIVES, not cumulative.
  Choose one mode per person:
  - Pauschale mode (default): 3% of Nettolohn, clamped to [2000, 4000].
  - Effektiv mode: sum of documented receipts (Physiosuisse, Fachliteratur,
    EDV-Hardware, etc.), only if the total EXCEEDS that person's Pauschale.
  Do NOT add individual receipts on top of the Pauschale.
  For married couples: decide mode per spouse independently, then sum.
  For weitereberufsauslagen.description, use exactly:
    "Berufsauslagen-Pauschale (3% des Nettolohns)"
  Do NOT append receipt names to the description.
- AHV/IV/EO/ALV contributions: field 9 MAIN line only, NOT NBU sub-line.
- Medical expenses (Krankheitskosten): DEDUCTIBLE ONLY above 5% of net income.
  STEP 1: Compute net income = total gross income − social insurance − BVG − professional expenses.
  STEP 2: Compute threshold = net_income × 0.05.
  STEP 3: If medical_total ≤ threshold, fill 0 (zero). Do NOT fill the raw receipt total.
  STEP 4: Only if medical_total > threshold, fill (medical_total − threshold).
""",

    "wealth": """## Wealth Section
- Bank balance: LAST line of bank statement CSV (Dec 31 balance).
- Bank name: read from the CSV metadata header (lines starting with '#' at top
  of file). Use the bank name as written there.
- Bank interest: sum ALL "Zinsabschluss" entries.
- Securities: market value as of Dec 31.
- Real estate: use Steuerwert from Gemeinde assessment, NOT market value.
- In Bank Accounts Include "_rows_needed": N if rows need adding.
- In Debts field creditor = "Hypothek" unless the property assessment or a loan document explicitly names the lender.
- Alimony received: declare the amount legally ordered per the Scheidungsurteil
     (decree), NOT the amount actually received in cash. If the decree specifies
     monthly amounts, multiply by the number of months the decree was in force
     during the tax year.

### Dynamic Rows
- Include "_rows_needed": N if rows need adding.
- If rows already exist, fill directly.
""",

    "attachments": "## Attachments\nFile uploads cannot be filled programmatically. Return {}.",
    "review": "## Review\nThis is the final review page. Return {}.",
}


def build_execute_user_message(page_name, fields, plan, doc_contents, guide_contents, filled_history):
    """Build the user message for the EXECUTE phase."""
    fillable = [f for f in fields if f.get("type") not in ("button", "text")]
    fields_desc = json.dumps(fillable, indent=2)

    docs_text = "\n\n".join(
        f"=== {name} ===\n{content[:8000]}"
        for name, content in doc_contents.items()
    )
    guides_text = "\n\n".join(
        f"=== {name} ===\n{content[:3000]}"
        for name, content in guide_contents.items()
    )

    history_lines = []
    for prev_page, prev_fields in filled_history.items():
        entries = ", ".join(f"{k}={v}" for k, v in prev_fields.items() if not k.startswith("_"))
        if entries:
            history_lines.append(f"  {prev_page}: {entries}")
    history_text = "\n".join(history_lines) if history_lines else "  (none yet)"

    plan_text = json.dumps(plan, indent=2) if isinstance(plan, dict) else str(plan)

    # ============================================================
    # STUDENT TODO: Assemble the user message.
    # Combine the variables above into a clear message with sections:
    #   - Your Plan (from the planning phase)
    #   - Document Contents (the actual data)
    #   - Tax Guides (if any)
    #   - Previously Filled
    #   - Current Page + Fields
    # ============================================================
    user_msg = ...  # <-- FILL THIS IN

    return user_msg

### Functions you'll use in `think_step` and `think_loop`

Here are the key functions you haven't seen yet. You'll call these in the cells below:

```python
# ── LLM calls ──
llm.chat_json(system, user, max_tokens=4096) -> dict
    # Like llm.chat(), but parses the response as a JSON dict.
    # Use this when you need structured output (plans, field mappings).

# ── Server tools ──
server.read_document(filename) -> dict
    # Returns {"content": "...", "filename": "..."}
    # Reads a document from the persona's folder.

server.fill_field(locator, value) -> dict
    # Returns {"success": True/False, ...}
    # Enters a value into the form field identified by locator.

# ── Helper (defined in next cell) ──
fill_page(server, mapping, page_name, filled_history) -> dict
    # Calls server.fill_field() for each {locator: value} in mapping.
    # Updates filled_history and returns the dict of successfully filled fields.
```


**Before you start coding `think_step`**, look back at the **architecture diagram** above. The function you're about to write implements all three boxes:

1. **Plan phase** — call the LLM with `PLAN_SYSTEM_PROMPT` + `build_plan_user_message()`
2. **Selective retrieval** — read only the documents the plan requested
3. **Execute phase** — call the LLM with `EXECUTE_SYSTEM_PROMPT` + `build_execute_user_message()`

Each phase is a separate `llm.chat_json()` call with a different system prompt.

In [ ]:
# ── THINK_STEP: Two-phase reasoning for a single page ─────────

def think_step(server, llm, page_name, elements, catalog, all_guides, filled_history, section_prompts):
    """Plan-then-Execute reasoning for one page.

    Phase 1 (Plan):   LLM sees field labels + doc catalog -> outputs a plan
    Phase 2 (Execute): LLM sees plan + selected doc contents -> outputs field mappings

    Returns
    -------
    tuple of (plan_dict, mapping_dict)
        plan_dict:    the plan JSON from phase 1
        mapping_dict: {locator: value} from phase 2
    """
    fields = [e for e in elements if e.get("type") not in ("button", "text")]
    if not fields:
        return {}, {}

    guides_list = list(all_guides.keys())

    # ════════════════════════════════════════════════════════════
    # PHASE 1: PLAN
    # ════════════════════════════════════════════════════════════
    # STUDENT TODO: Call the LLM to create a plan.
    # Use: llm.chat_json(), with the correct parameters
    plan = ...  # <-- FILL THIS IN

    print(f"  Plan: need docs {plan.get('documents_needed', [])}")
    print(f"  Strategy: {plan.get('strategy', 'none')}")

    # ════════════════════════════════════════════════════════════
    # SELECTIVE RETRIEVAL
    # ════════════════════════════════════════════════════════════
    # STUDENT TODO: Read only the documents the plan requested.
    doc_contents = {}
    for doc_name in plan.get("documents_needed", []):
        ...  # <-- FILL IN: call server.read_document(), store content

    # STUDENT TODO: Fetch only the guides the plan requested.
    guide_contents = {}
    for guide_name in plan.get("guides_needed", []):
        ...  # <-- FILL IN: look up in all_guides dict

    # ════════════════════════════════════════════════════════════
    # PHASE 2: EXECUTE
    # ════════════════════════════════════════════════════════════
    section = page_name.split("/")[0]
    # STUDENT TODO: Call the LLM to get field mappings.
    # Use: llm.chat_json(), system_msg, build_execute_user_message()
    # Hint: What should the system prompt be?
    section_prompt = ... # <-- FILL THIS IN
    system_msg = ... # <-- FILL THIS IN
    mapping = ...  # <-- FILL THIS IN

    return plan, mapping

In [ ]:
# Test think_step on the Income page (MockBridge)

from mcp_server.bridges.mock import MockBridge

test_bridge = MockBridge()
test_bridge.frontend.current_page = "income"

test_server = MCPServer(
    persona_folder=f"{REPO}/personas/{PERSONA}",
    guides_folder=f"{REPO}/guides",
    bridge=test_bridge,
)

test_catalog = build_document_catalog(test_server)

def load_all_guides(guides_folder):
    guides = {}
    folder = Path(guides_folder)
    if not folder.exists():
        return guides
    for f in sorted(folder.iterdir()):
        if f.suffix in (".html", ".txt", ".md"):
            guides[f.stem] = f.read_text(encoding="utf-8")
    return guides

all_guides = load_all_guides(f"{REPO}/guides")

# ============================================================
# STUDENT TODO: Get the current page data from the server.
# Which tool from Section 5 returns all fields on the current page?
# ============================================================
page_data = ...  # <-- FILL THIS IN (hint: use test_server)

fillable = [e for e in page_data["elements"] if e.get("type") not in ("button", "text")]

print(f"Testing think_step on: {page_data['page_name']} ({len(fillable)} fillable fields)")

plan, mapping = think_step(
    server=test_server,
    llm=llm,
    page_name=page_data["page_name"],
    elements=page_data["elements"],
    catalog=test_catalog,
    all_guides=all_guides,
    filled_history={},
    section_prompts=SECTION_PROMPTS,
)

print(f"Plan returned {len(plan.get('documents_needed', []))} documents")
print(f"Mapping returned {len([k for k in mapping if not k.startswith('_')])} fields")

In [ ]:
#@title Display think_step results & ground truth comparison { display-mode: "form" }
from IPython.display import display, HTML

# Display field mappings
thought = mapping.get("_thought", "")
rows = "".join(
    f'<tr><td><code>{k}</code></td><td>{v}</td></tr>'
    for k, v in mapping.items() if not k.startswith("_")
)
display(HTML(f'''
<div style="border:2px solid #2d7d2d; border-radius:8px; padding:14px; margin:10px 0;
            background:#f0f8f0; font-family:sans-serif;">
    <h4 style="color:#2d7d2d; margin:0 0 8px 0;">Field Mappings from LLM</h4>
    {"<div style='margin-bottom:8px;'><b>Reasoning:</b> " + thought + "</div>" if thought else ""}
    <table style="border-collapse:collapse; width:100%;">
    <tr style="background:#2d7d2d; color:white;">
        <th style="padding:6px 8px; text-align:left;">Locator</th>
        <th style="padding:6px 8px; text-align:left;">Value</th>
    </tr>
    {rows if rows else "<tr><td colspan='2'><i>No mappings returned</i></td></tr>"}
    </table>
</div>
'''))

# Ground truth comparison
gt_path = Path(f"{REPO}/personas/{PERSONA}/ground_truth.json")
if gt_path.exists():
    with open(gt_path) as f:
        ground_truth = json.load(f)

    def flatten_gt(section_data, prefix="field-income"):
        flat = {}
        for key, val in section_data.items():
            if isinstance(val, dict):
                for subkey, subval in val.items():
                    flat[f"{prefix}-{key}-{subkey}"] = str(subval)
            else:
                flat[f"{prefix}-{key}"] = str(val)
        return flat

    gt_income = flatten_gt(ground_truth.get("income", {}))
    agent_values = {k: str(v) for k, v in mapping.items() if not k.startswith("_")}

    all_locators = sorted(set(list(gt_income.keys()) + list(agent_values.keys())))
    comp_rows = ""
    for loc in all_locators:
        expected = gt_income.get(loc, "")
        actual = agent_values.get(loc, "")
        if expected and actual:
            match = abs(float(expected) - float(actual)) < 1 if expected.replace(".", "").isdigit() and actual.replace(".", "").isdigit() else expected.lower() == actual.lower()
            color = "#2d7d2d" if match else "#cc0000"
            icon = "&#10003;" if match else "&#10007;"
        elif expected and not actual:
            color = "#cc6600"
            icon = "&#8709;"
        else:
            color = "#999"
            icon = "?"
        comp_rows += (
            f'<tr style="color:{color};">'
            f'<td style="padding:4px 8px;"><code>{loc}</code></td>'
            f'<td style="padding:4px 8px;">{expected or "<i>-</i>"}</td>'
            f'<td style="padding:4px 8px;">{actual or "<i>not filled</i>"}</td>'
            f'<td style="padding:4px 8px; text-align:center;">{icon}</td></tr>'
        )

    display(HTML(f'''
    <div style="border:2px solid #4a90d9; border-radius:8px; padding:14px; margin:10px 0;
                background:linear-gradient(135deg,#f8faff 0%,#eef3ff 100%); font-family:sans-serif;">
        <h4 style="color:#4a90d9; margin:0 0 8px 0;">Ground Truth Comparison (Income)</h4>
        <table style="border-collapse:collapse; width:100%;">
        <tr style="background:#4a90d9; color:white;">
            <th style="padding:6px 8px; text-align:left;">Locator</th>
            <th style="padding:6px 8px; text-align:left;">Expected</th>
            <th style="padding:6px 8px; text-align:left;">Agent</th>
            <th style="padding:6px 8px; text-align:center;">Match</th>
        </tr>
        {comp_rows}
        </table>
    </div>
    '''))
else:
    print("Ground truth file not found — skipping comparison.")

**Checkpoint:** If `think_step` returned reasonable values for the income page (e.g. gross salary around 142,000 for Anna Meier), you're ready to visualize the plan and build the full loop.

If the values look wrong, check:
- Is `build_plan_user_message` sending the document catalog correctly?
- Is `build_execute_user_message` including the actual document content?
- Is the plan requesting the right documents?

## 7. Plan Visualization

One advantage of the Plan-Execute split: we can **visualize** the agent's decision-making before it acts. This makes the agent's reasoning transparent and debuggable.

In [ ]:
#@title visualize_plan() — renders plan as HTML card { display-mode: "form" }
# Pre-built visualization function
from IPython.display import display, HTML

def visualize_plan(page_name, plan, doc_catalog):
    """Render the agent's plan as a styled HTML card."""
    selected = set(plan.get("documents_needed", []))

    docs_html = ""
    for doc in doc_catalog:
        is_selected = doc["filename"] in selected
        icon = "&#9745;" if is_selected else "&#9744;"
        color = "#2d7d2d" if is_selected else "#999"
        weight = "bold" if is_selected else "normal"
        docs_html += (
            f'<div style="color:{color}; font-weight:{weight}; margin:2px 0;">'
            f'{icon} <code>{doc["filename"]}</code> — {doc["description"]}</div>'
        )

    fields_html = ""
    for f in plan.get("fields_to_fill", []):
        fields_html += f'<code style="background:#e8f5e9; padding:2px 6px; margin:2px; border-radius:3px;">{f}</code> '

    reasoning = plan.get("_reasoning", "No reasoning provided")
    strategy = plan.get("strategy", "No strategy provided")

    html = f"""
    <div style="border:2px solid #4a90d9; border-radius:10px; padding:16px; margin:12px 0;
                background:linear-gradient(135deg, #f8faff 0%, #eef3ff 100%); font-family:sans-serif;">
        <h3 style="color:#4a90d9; margin-top:0;">Plan: {page_name}</h3>
        <div style="background:white; border-radius:6px; padding:12px; margin:8px 0;">
            <b>Reasoning:</b>
            <blockquote style="border-left:3px solid #4a90d9; padding-left:12px; color:#333; margin:8px 0;">
                {reasoning}
            </blockquote>
        </div>
        <div style="background:white; border-radius:6px; padding:12px; margin:8px 0;">
            <b>Documents selected:</b><br/>
            {docs_html}
        </div>
        <div style="background:white; border-radius:6px; padding:12px; margin:8px 0;">
            <b>Strategy:</b> {strategy}
        </div>
        <div style="background:white; border-radius:6px; padding:12px; margin:8px 0;">
            <b>Fields to fill:</b><br/>{fields_html if fields_html else "<i>not specified</i>"}
        </div>
    </div>
    """
    display(HTML(html))

print("visualize_plan() defined")

In [ ]:
#@title Visualize the plan from the test { display-mode: "form" }
if plan:
    visualize_plan(page_data["page_name"], plan, test_catalog)
else:
    print("No plan to visualize yet. Run the think_step test cell first.")

## 8. Think Loop — Full Pipeline

Now we wire `think_step` into the full navigation loop. The code drives page-by-page navigation; your `think_step` handles the reasoning on each page.

The loop: **scan** → **plan** → **retrieve** → **execute** → **fill** → **next page** → repeat until review → **submit**.

In [ ]:
#@title Helper: add dynamic rows { display-mode: "form" }

def add_rows_if_needed(server, elements, num_rows_needed=0):
    """Click 'Add Row' buttons to create dynamic table rows."""
    add_btns = [e for e in elements
                if e.get("type") == "button"
                and "add-row" in (e.get("locator") or "").lower()]
    if not add_btns:
        return None

    existing_rows = set()
    for e in elements:
        loc = e.get("locator") or ""
        match = re.search(r'-(\d+)-', loc)
        if match:
            existing_rows.add(int(match.group(1)))

    rows_to_add = max(0, (num_rows_needed or 1) - len(existing_rows))
    if rows_to_add > 0:
        for _ in range(rows_to_add):
            for btn in add_btns:
                loc = btn.get("locator")
                if loc:
                    print(f"  Adding row: {loc}")
                    server.click_element(loc)
        return server.scan_page()
    return None


# ── Helper: fill fields from mapping ─────────────────────────

def fill_page(server, mapping, page_name, filled_history):
    """Execute fill_field for each locator:value in the mapping."""
    filled = {}
    thought = mapping.pop("_thought", None)
    if thought:
        print(f"  Reasoning: {thought}")

    for locator, value in mapping.items():
        if locator.startswith("_"):
            continue
        result = server.fill_field(locator, str(value))
        if result.get("success"):
            filled[locator] = value
            print(f"  Filled: {locator} = {value}")
        else:
            print(f"  FAILED: {locator}: {result.get('error', 'unknown')}")

    if filled:
        filled_history[page_name] = filled
    return filled

In [ ]:
# ── Visualization helpers for think_loop ──────────────────────

from IPython.display import display, HTML

def display_page_header(step, page_name):
    """Show a styled header for the current page."""
    display(HTML(f'''
    <div style="border-left:4px solid #4a90d9; padding:8px 14px; margin:16px 0 8px 0;
                background:#eef3ff; font-family:sans-serif;">
        <b style="color:#4a90d9;">Page {step}: {page_name}</b>
    </div>
    '''))

def display_submit_result(result):
    """Show submission success/failure."""
    display(HTML(f'''
    <div style="border:2px solid #2d7d2d; border-radius:8px; padding:14px; margin:10px 0;
                background:#f0f8f0; font-family:sans-serif;">
        <h4 style="color:#2d7d2d; margin:0;">Submitted! Success: {result.get("success")}</h4>
    </div>
    '''))

def display_skip_message():
    """Show message when no fillable fields found."""
    display(HTML('<div style="color:#999; margin:4px 0; font-family:sans-serif;"><i>No fillable fields — skipping</i></div>'))

def display_field_count(count):
    """Show the number of fillable fields."""
    display(HTML(f'<div style="margin:4px 0; font-family:sans-serif;"><b>{count}</b> fillable fields</div>'))

def display_filled_count(count):
    """Show how many fields were filled."""
    display(HTML(f'<div style="color:#2d7d2d; font-family:sans-serif;"><b>{count}</b> fields filled</div>'))

def display_empty_mapping():
    """Show message when LLM returns empty mapping."""
    display(HTML('<div style="color:#999; font-family:sans-serif;"><i>LLM returned empty mapping — nothing to fill</i></div>'))

def display_navigation(target):
    """Show navigation message."""
    display(HTML(f'<div style="color:#4a90d9; font-family:sans-serif;">&#8594; Navigated to: <b>{target}</b></div>'))

def display_nav_error():
    """Show navigation failure."""
    display(HTML('<div style="color:#cc0000; font-family:sans-serif;"><b>Cannot navigate further — stopping</b></div>'))

def display_max_steps(max_steps):
    """Show max steps warning."""
    display(HTML(f'<div style="color:#cc6600; font-family:sans-serif; margin-top:12px;"><b>Max steps ({max_steps}) reached without submitting.</b></div>'))

print("Visualization helpers defined")

In [ ]:
# THE FULL THINK LOOP

def think_loop(server, llm, max_steps=100):
    """Full agent: Plan-Execute on every page with visualization."""

    catalog = build_document_catalog(server)
    all_guides = load_all_guides(f"{REPO}/guides")
    filled_history = {}
    plans = []
    total_fields = 0

    # Auto-navigate away from root/overview
    initial = server.scan_page()
    page_name = initial.get("page_name", "")
    if page_name in ("root", "overview", ""):
        for loc in ["tab-nav-form", "btn-nav-personal", "btn-login"]:
            r = server.click_element(loc)
            if r.get("success") and r.get("new_page"):
                break

    visited = set()
    step = 0

    while step < max_steps:
        step += 1
        page_data = server.scan_page()
        page_name = page_data.get("page_name", "unknown")
        elements = page_data.get("elements", [])

        # Skip non-form pages
        if page_name in ("root", "overview", ""):
            server.click_element("tab-nav-form")
            continue

        # Skip already-visited pages
        if page_name in visited:
            nav = server.click_element("btn-next")
            if not nav.get("new_page"):
                break
            continue
        visited.add(page_name)

        display_page_header(step, page_name)

        # Submit on review page
        if page_name == "review":
            result = server.submit_form()
            display_submit_result(result)
            return {
                "submission": result,
                "filled_history": filled_history,
                "plans": plans,
            }

        # Handle dynamic rows
        fillable = [e for e in elements if e.get("type") not in ("button", "text")]
        has_add_row = any(
            e.get("type") == "button" and "add-row" in (e.get("locator") or "").lower()
            for e in elements
        )
        if not fillable and has_add_row:
            new_scan = add_rows_if_needed(server, elements)
            if new_scan:
                elements = new_scan.get("elements", [])

        fillable = [e for e in elements if e.get("type") not in ("button", "text")]
        if not fillable:
            display_skip_message()
        else:
            display_field_count(len(fillable))

            # ════════════════════════════════════════════════════
            # STUDENT TODO: Call think_step and fill the page
            # 1. Call think_step() to get (plan, mapping)
            # 2. Visualize the plan
            # 3. Handle _rows_needed if present
            # 4. Fill the page
            # ════════════════════════════════════════════════════

            # Step 1: Get the plan and mapping
            plan, mapping = ...  # <-- FILL IN: call think_step()

            # Step 2: Visualize
            if plan:
                plans.append({"page": page_name, "plan": plan})
                visualize_plan(page_name, plan, catalog)

            # Step 3: Handle _rows_needed
            rows_needed = int(mapping.pop("_rows_needed", 0))
            if rows_needed > 0:
                new_scan = add_rows_if_needed(server, elements, rows_needed)
                if new_scan:
                    elements = new_scan.get("elements", [])
                    _, mapping = think_step(
                        server, llm, page_name, elements,
                        catalog, all_guides, filled_history, SECTION_PROMPTS,
                    )
                    mapping.pop("_rows_needed", None)

            # Step 4: Fill the page
            if mapping:
                ...  # <-- FILL IN: call fill_page()
            else:
                display_empty_mapping()

        # Navigate to next page
        nav_result = server.click_element("btn-next")
        if nav_result.get("success") and nav_result.get("new_page"):
            display_navigation(nav_result["new_page"])
        else:
            for nav_target in ["tab-nav-form", "nav-personal", "nav-income",
                               "nav-deductions", "nav-wealth",
                               "nav-attachments", "nav-review"]:
                r = server.click_element(nav_target)
                if r.get("success") and r.get("new_page"):
                    display_navigation(r["new_page"])
                    break
            else:
                display_nav_error()
                break

    display_max_steps(max_steps)
    return {"submission": None, "filled_history": filled_history, "plans": plans}

In [ ]:
# ════════════════════════════════════════════════════════════════
# THE BIG CELL — Display UI + Init Server + Run Agent
# ════════════════════════════════════════════════════════════════
# This cell MUST be a single cell because Colab's eval_js()
# only works within the same output cell where HTML was rendered.
# ════════════════════════════════════════════════════════════════

# -- Change PERSONA here! --
PERSONA = "anna_meier"
# Available: 'anna_meier' (Easy), 'marco_laura_bernasconi' (Medium),
#            'priya_chakraborty' (Hard), 'thomas_elisabeth_widmer' (Very Hard),
#            'yuki_tanaka' (Expert)

# 1. Render the UI
from IPython.display import display, HTML
import os, json, datetime
from google.colab.output import eval_js
import time

BUILD_DIR = f"{REPO}/EnvDemoV1/dist"
with open(os.path.join(BUILD_DIR, "index.html")) as f:
    html_content = f.read()

html_content = html_content.replace('"\/assets/', f'"http://localhost:{PORT}/assets/')
html_content = html_content.replace('"./assets/', f'"http://localhost:{PORT}/assets/')
html_content = html_content.replace('"/assets/', f'"http://localhost:{PORT}/assets/')

display(HTML(html_content))
print(f"UI rendered from http://localhost:{PORT}")

time.sleep(1)
eval_js(f'window.TaxPortal.setPersona("{PERSONA}")')

# 2. Init MCP Server with ColabBridge
from mcp_server.server import MCPServer
from mcp_server.bridges.colab import ColabBridge

server = MCPServer(
    persona_folder=f"{REPO}/personas/{PERSONA}",
    guides_folder=f"{REPO}/guides",
    bridge=ColabBridge(),
)
print(f"Server ready for: {PERSONA}")

# 3. Run the agent
print(f"\nStarting agent for {PERSONA}...\n")
agent_result = think_loop(server, llm, max_steps=100)
print("\nDone!")

# 4. Save outputs
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
out_dir = f"/content/agent_output_{PERSONA}_{timestamp}"
os.makedirs(out_dir, exist_ok=True)

with open(f"{out_dir}/filled_history.json", "w") as f:
    json.dump(agent_result.get("filled_history", {}), f, indent=2, ensure_ascii=False)
with open(f"{out_dir}/plans.json", "w") as f:
    json.dump(agent_result.get("plans", []), f, indent=2, ensure_ascii=False, default=str)
with open(f"{out_dir}/submission.json", "w") as f:
    json.dump(agent_result.get("submission", {}), f, indent=2, ensure_ascii=False, default=str)

print(f"Output saved to: {out_dir}")

In [ ]:
#@title Scoring — Install dependencies { display-mode: "form" }
!pip install rapidfuzz

In [ ]:
#@title Scoring — Hybrid Scoring { display-mode: "form" }
import json, re
from pathlib import Path
from IPython.display import display, HTML
import numpy as np
from rapidfuzz import fuzz

# Helper functions for scoring

def normalize(s):
    """Lowercase, strip punctuation/whitespace."""
    s = str(s).lower().strip()
    return re.sub(r'[^\w\s]', '', s)

# Fuzzy matching
def fuzzy_score(a, b):
    """Token-sort ratio ∈ [0, 1] via rapidfuzz."""
    return fuzz.token_sort_ratio(str(a), str(b)) / 100.0

def cosine_sim(a, b):
    """Cosine similarity ∈ [-1, 1]."""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9)

def try_numeric(s):
    return float(str(s).replace(",", "").replace("'", "").strip())

# LLM call to look into cases where semantic meaning might still find correctness
def llm_score(field_name: str, sub_val: str, gt_val: str, llm) -> float:
    prompt = f"""You are scoring an AI-filled tax form field.
      Field: {field_name}
      Submitted answer: {sub_val}
      Expected answer: {gt_val}

      Are these equivalent for the purposes of a Swiss tax form? 
      For numeric/amount fields, require exact or near-exact match.
      For description/name fields, score semantically.

      Reply with ONLY one of: 0, 0.5, 1
      - 1 = correct (same value, different format is fine)
      - 0.5 = partially correct or ambiguous
      - 0 = wrong"""
    try:
        resp = llm.chat(system="You are a precise tax form evaluator.", 
                        user=prompt, max_tokens=5, temperature=0)
        return float(resp.strip())
    except:
        return 0.5  # neutral fallback on failure

def hybrid_score(sub_val, gt_val, field_name: str, llm=None,
                 w_exact=0.4, w_norm=0.25, w_fuzzy=0.25):
    """
    Weighted hybrid: exact + normalized + fuzzy + embedding.
    Weights sum to 1.0. Embedding component dropped (redistributed)
    if api_key is None.
    """
    sv, gv = str(sub_val).strip(), str(gt_val).strip()

    exact  = 1.0 if sv.lower() == gv.lower() else 0.0
    norm   = 1.0 if normalize(sv) == normalize(gv) else 0.0
    fuzzy  = fuzzy_score(sv, gv)
    
    # Redistribute embedding weight proportionally
    total_w = w_exact + w_norm + w_fuzzy
    h_lexical   = (w_exact*exact + w_norm*norm + w_fuzzy*fuzzy) / total_w

    if llm is not None:
        llm_s = llm_score(field_name, sub_val, gt_val, llm)
        # Blend: 70% lexical, 30% LLM
        return 0.7 * h_lexical + 0.3 * llm_s
    return h_lexical

MATCH_THRESHOLD = 0.75  # tune as needed

# Main scoring logic

def score_submission(repo, persona, agent_result):
    gt_path = Path(f"{repo}/personas/{persona}/ground_truth.json")
    if not gt_path.exists():
        display(HTML(f'<div style="color:#cc0000;"><b>No ground_truth.json for {persona}</b></div>'))
        return

    with open(gt_path) as f:
        ground_truth = json.load(f)

    submission = agent_result.get("submission", {})
    submission_data = submission.get("submission_json", submission) if isinstance(submission, dict) else {}

    counts = {"correct": 0, "wrong": 0, "missing": 0, "total": 0}
    wrong_rows = []
    missing_rows = []

    def compare_fields(gt, sub, path=""):
        if not isinstance(gt, dict):
            return
        for key, gt_val in gt.items():
            if key.startswith("_"):
                continue
            full_key = f"{path}.{key}" if path else key
            if isinstance(gt_val, dict):
                compare_fields(gt_val, sub.get(key, {}) if isinstance(sub, dict) else {}, full_key)
            elif isinstance(gt_val, list):
                sub_list = sub.get(key, []) if isinstance(sub, dict) else []
                if not isinstance(sub_list, list):
                    sub_list = []
                for i, gt_item in enumerate(gt_val):
                    if i >= len(sub_list):
                        # Entire missing list item — count all its leaf fields as missing
                        counts["total"] += 1
                        counts["missing"] += 1
                        missing_rows.append(f'<tr style="background:#fff3e0;"><td><code>{full_key}[{i}]</code></td><td>—</td><td>(entire item missing)</td></tr>')
                    else:
                        sub_item = sub_list[i]
                        compare_fields(gt_item, sub_item if isinstance(sub_item, dict) else {}, f"{full_key}[{i}]")
            else:
                if gt_val is None or gt_val == "" or gt_val != gt_val:  # last: NaN guard
                    continue
                counts["total"] += 1
                sub_val = sub.get(key) if isinstance(sub, dict) else None
                if sub_val is None or sub_val == "":
                    counts["missing"] += 1
                    missing_rows.append(f'<tr style="background:#fff3e0;"><td><code>{full_key}</code></td><td>—</td><td>{gt_val}</td></tr>')
                else:
                    # Numeric fast-path (unchanged behaviour for numbers)
                    try:
                        nv, ng = try_numeric(sub_val), try_numeric(gt_val)
                        numeric_match = abs(nv - ng) < 1.0
                    except (ValueError, TypeError):
                        numeric_match = False

                    if numeric_match:
                        counts["correct"] += 1
                    else:
                        h = hybrid_score(sub_val, gt_val, full_key, llm)
                        if h >= MATCH_THRESHOLD:
                            counts["correct"] += 1
                        else:
                            counts["wrong"] += 1
                            wrong_rows.append(
                                f'<tr style="background:#ffebee;">'
                                f'<td><code>{full_key}</code></td>'
                                f'<td>{sub_val} <span style="color:#888;font-size:0.85em;">(h={h:.2f})</span></td>'
                                f'<td>{gt_val}</td></tr>'
                            )

    compare_fields(ground_truth, submission_data)

    correct, wrong, missing, total = counts["correct"], counts["wrong"], counts["missing"], counts["total"]
    score = (correct / total * 100) if total > 0 else 0
    score_color = "#2d7d2d" if score >= 80 else "#cc6600" if score >= 50 else "#cc0000"

    error_table = ""
    if wrong_rows or missing_rows:
        error_table = '<table style="border-collapse:collapse; width:100%; margin-top:10px;"><tr style="background:#666; color:white;"><th style="padding:6px 8px; text-align:left;">Field</th><th style="padding:6px 8px; text-align:left;">Got</th><th style="padding:6px 8px; text-align:left;">Expected</th></tr>' + "".join(wrong_rows) + "".join(missing_rows) + '</table>'

    display(HTML(f'<div style="border:2px solid {score_color}; border-radius:10px; padding:16px; margin:12px 0; font-family:sans-serif;"><h3 style="color:{score_color}; margin:0 0 10px 0;">Score: {correct}/{total} ({score:.1f}%)</h3><div style="display:flex; gap:20px; margin-bottom:8px;"><span style="color:#2d7d2d;"><b>{correct}</b> Correct</span><span style="color:#cc0000;"><b>{wrong}</b> Wrong</span><span style="color:#cc6600;"><b>{missing}</b> Missing</span></div><div style="background:#eee; border-radius:6px; height:20px; overflow:hidden; margin:8px 0;"><div style="background:{score_color}; height:100%; width:{score:.0f}%; border-radius:6px;"></div></div>{error_table}</div>'))

score_submission(REPO, PERSONA, agent_result)


## 9. Wrap-Up

### What you built

An AI agent that autonomously files a Swiss tax return using a **Plan-then-Execute** architecture:

1. **Plan phase**: The LLM sees document *names*, not content, and decides what to read
2. **Selective retrieval**: Only the documents the plan requested are loaded
3. **Execute phase**: The LLM fills fields using only the relevant documents
4. **Visualization**: Every plan is rendered as a visual card showing the agent's reasoning

### Key insights

- **Selective retrieval matters**: Sending only relevant documents reduces noise and keeps context focused. This is especially important for local LLMs with smaller context windows.
- **Planning makes agents transparent**: By splitting reasoning into plan + execute, you can see *why* the agent made each decision — not just what it did.
- **The agent doesn't see the whole form at once**: It works page by page, building up context through `filled_history`.

### Extensions to explore
- Add a **validation loop**: after filling a page, scan for validation errors and re-ask the LLM
- Use `ask_user` to handle ambiguous fields (the NPC taxpayer has verbal knowledge not in documents)
- Try different local models and compare accuracy vs speed
- Build a **confidence score** into the plan — if the LLM is uncertain, flag it for human review

### References
- [OpenRouter Documentation](https://openrouter.ai/docs)
- [OpenAI Chat Completions API](https://platform.openai.com/docs/api-reference/chat)
- [Swiss Tax System Overview](https://www.estv.admin.ch/)
- [MCP Protocol Specification](https://modelcontextprotocol.io/)